In [1]:
!rm -rf diploma_centpy_parallelization_py
# Флаг -b указывает конкретную ветку
!git clone -b feature/jax-centpy https://github.com/filkinc/diploma_centpy_parallelization_py.git
%cd diploma_centpy_parallelization_py
%cd /content/diploma_centpy_parallelization_py/jax_centpy

# Установка зависимостей
!pip install centpy pandas matplotlib seaborn

Cloning into 'diploma_centpy_parallelization_py'...
remote: Enumerating objects: 268, done.
remote: Counting objects: 100% (53/53), done.
remote: Compressing objects: 100% (46/46), done.
remote: Total 268 (delta 8), reused 24 (delta 7), pack-reused 215 (from 1)
Receiving objects: 100% (268/268), 22.38 MiB | 22.92 MiB/s, done.
Resolving deltas: 100% (102/102), done.
/content/diploma_centpy_parallelization_py
/content/diploma_centpy_parallelization_py/jax_centpy


In [2]:
import os
import time
import jax
import jax.numpy as jnp
jax.config.update("jax_enable_x64", True)
import matplotlib.pyplot as plt
import matplotlib.animation as animation
from IPython.display import HTML
import numpy as np
from google.colab import files
from typing import Callable, NamedTuple

from core import Pars2d, Equation2d, Pars1d, Equation1d
from solver import Solver1d, FastSolver1d
from boundaries import periodic_bc_2d, neumann_bc_2d, dirichlet_riemann_bc_2d
from equations import make_euler_riemann_2d, make_euler_isentropic_vortex_2d
from schemes import compute_rhs_sd2_2d
from limiters import monotonized_central, minmod
from richardson import self_convergence_analysis

In [6]:
def make_modified_sod(gamma=1.4, alpha: float = 0.5) -> Equation1d:
    def compute_pressure(q):
        rho = q[..., 0]
        u = q[..., 1] / rho
        E = q[..., 2]
        return (gamma - 1.0) * (E - 0.5 * rho * u**2)

    def flux(q):
        rho = q[..., 0]
        rhou = q[..., 1]
        E = q[..., 2]
        rhosafe = jnp.maximum(rho, 1e-10)
        u = rhou / rhosafe
        p = compute_pressure(q)
        return jnp.stack([rhou, rhou * u + p, u * (E + p)], axis=-1)

    def spectral_radius(q):
        rho = q[..., 0]
        rhou = q[..., 1]
        rhosafe = jnp.maximum(rho, 1e-10)
        u = rhou / rhosafe
        p = jnp.maximum(compute_pressure(q), 1e-10)
        c = jnp.sqrt(gamma * p / rhosafe)
        a = jnp.abs(u) + c
        return a[..., None]

    def initial_data(x):
        # Невозмущенный газ
        rho_0, p_0 = 1.0, 1.0
        # alpha = 0.1 # Коэффициент понижения плотности в каверне

        # Состояние за ударной волной (по Ренкину-Гюгонио для M=2)
        rho_R = 2.66666666667
        p_R = 4.5
        u_R = -1.47901994677

        # Границы областей
        x_0 = 0.3
        x_1 = 0.5
        x_sw = 0.8

        # Кусочно-постоянное распределение плотности
        rho = jnp.where(x < x_0, rho_0,
                jnp.where(x <= x_1, alpha * rho_0,
                jnp.where(x < x_sw, rho_0, rho_R)))

        # Распределение скорости (движется только газ за фронтом УВ)
        u = jnp.where(x < x_sw, 0.0, u_R)

        # Распределение давления
        p = jnp.where(x < x_0, p_0,
                jnp.where(x <= x_1, p_0,
                jnp.where(x < x_sw, p_0, p_R)))

        E = p / (gamma - 1.0) + 0.5 * rho * u**2
        return jnp.stack([rho, rho * u, E], axis=-1)

    def neumann_bc(u, nghost):
        # Граничные условия Неймана: дублирование крайних ячеек (нулевой градиент)
        return jnp.pad(u, ((nghost, nghost), (0, 0)), mode='edge')

    return Equation1d(
        flux=flux,
        spectral_radius=spectral_radius,
        initial_data=initial_data,
        boundary_handler=neumann_bc,
        name="Modified Sod with Neumann BC"
    )

In [7]:
# Инициализация уравнения и параметров
eqn = make_modified_sod()
pars = Pars1d(x_init=0.0, x_final=1.0, t_final=0.25, dt_out=0.005, J=400, cfl=0.45, scheme="sd2")
solver = Solver1d(pars, eqn, scheme_name='sd2', limiter_name='minmod')

print("Запуск расчёта...")
sol = solver.solve()
print("Расчёт успешно завершён!")

# Извлечение результатов
x_num = np.array(sol["x"])
t_num = np.array(sol["t"])
q_num = np.array(sol["u_n"])

# Декодирование консервативных переменных в примитивные
rho_num = q_num[..., 0]
u_num = q_num[..., 1] / q_num[..., 0]
p_num = 0.4 * (q_num[..., 2] - 0.5 * rho_num * u_num**2)

Запуск расчёта...
Starting simulation: Modified Sod with Neumann BC
Grid: 400 points, Scheme: SD2/minmod
Simulation finished in 1.0972s
Total steps: 766
Расчёт успешно завершён!


In [8]:
fig, axes = plt.subplots(3, 1, figsize=(8, 10))
fig.tight_layout(pad=4.0)

# Настройка графика плотности
line_rho, = axes[0].plot([], [], 'r-')
axes[0].set_xlim(0, 1)
axes[0].set_ylim(0, 3.5)
axes[0].set_ylabel('Плотность (Density)')
axes[0].grid(True)

# Настройка графика скорости
line_u, = axes[1].plot([], [], 'r-')
axes[1].set_xlim(0, 1)
axes[1].set_ylim(-2.0, 1.0)
axes[1].set_ylabel('Скорость (Velocity)')
axes[1].grid(True)

# Настройка графика давления
line_p, = axes[2].plot([], [], 'r-')
axes[2].set_xlim(0, 1)
axes[2].set_ylim(0, 5)
axes[2].set_ylabel('Давление (Pressure)')
axes[2].set_xlabel('Координата x')
axes[2].grid(True)

title = fig.suptitle('', fontsize=14)

def init():
    line_rho.set_data([], [])
    line_u.set_data([], [])
    line_p.set_data([], [])
    title.set_text('')
    return line_rho, line_u, line_p, title

def animate(i):
    line_rho.set_data(x_num, rho_num[i])
    line_u.set_data(x_num, u_num[i])
    line_p.set_data(x_num, p_num[i])
    title.set_text(f'Модифицированная задача Сода | Время: {t_num[i]:.3f}')
    return line_rho, line_u, line_p, title

anim = animation.FuncAnimation(
    fig, animate, init_func=init,
    frames=len(t_num), interval=50, blit=True
)

plt.close(fig) # Закрываем статичную фигуру, чтобы она не дублировалась
HTML(anim.to_jshtml()) # Отображаем интерактивный плеер